# Sprint 3: Enterprise Knowledge Assistant

### Business Requirement:

Our AI assistant should answer the questions  from our Employee Handbook PDF using complete RAG workflow  without using Langchain.

# ================================================================

### 1.Install Required Libraries

In [1]:
pip install -q openai pypdf chromadb

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2.Configure OpenAI API Key

In [6]:
import os
from getpass import  getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key:")

from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
print("OpenAI Client configured successfully.")

OpenAI Client configured successfully.


### 3.Upload the  PDF

In [7]:
import os

pdf_path = r"D:\ProITBridge\Courses\Gen AI and Agentic AI\AI Assistant Appication\PROITBRIDGE_Employee_Handbook_2026.pdf"

print("Uploaded file:", pdf_path)
print("File exists:", os.path.exists(pdf_path))

Uploaded file: D:\ProITBridge\Courses\Gen AI and Agentic AI\AI Assistant Appication\PROITBRIDGE_Employee_Handbook_2026.pdf
File exists: True


### 4.PDF  Reader

In [8]:
from pypdf import PdfReader
reader = PdfReader(pdf_path)

pages=[]

for page_number,page in enumerate(reader.pages,start=1):
    page_text =page.extract_text() or ""
    pages.append({
        "page":page_number,
        "text":page_text
    })

text = "\n".join(page["text"] for  page in pages)

print("Pages:",len(pages))
print("Characters Extracted:",len(text))
print("\nPreview:\n")
print(text[:2000])

Pages: 57
Characters Extracted: 104734

Preview:

PEOPLE · POLICIES · PRACTICES
Employee
Handbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applicable law
CLASSIFICATION Internal — Confidential
PROITBRIDGE TECHNOLOGIES PRIVATE LIMITED PIB-HR-HB-2026-V6
CONTENTS
Table of Contents
This handbook is organised into 22 chapters. Use the page numbers below to navigate to the policy
you need.
01 Welcome to PROITBRIDGE
A message from the Founder & CEO
4
02 About the Company
Who we are, vision, mission, values, structure
6
03 How to Use This Handbook
Scope, authority, amendments, disclaimer
9
04 Empl

## 5.Chunking

For this eg, we are using a "Simple Character-based chunking" strategy. In a production system,chunking can be made more sphisticated using paragraphs,sections,headings,token counts or semantic boundaries

In [11]:
chunk_size = 500
overlap = 50

chunks=[]
start =0
chunk_id =0

while start < len(text):
    end = start  + chunk_size
    chunk_text=text[start:end].strip()

    if chunk_text:
        chunks.append({
            "id":f"chunk-{chunk_id}",
            "text":chunk_text
        })
        chunk_id+=1

    start +=chunk_size-overlap
print("Total Chunks:",len(chunks))
print("\nFirst Chunk:\n")
print(chunks[0]["text"])

Total Chunks: 233

First Chunk:

PEOPLE · POLICIES · PRACTICES
Employee
Handbook
Everything you need to know about working at
PROITBRIDGE — how we hire, how we pay, how we grow,
how we protect our people and our clients, and what we
expect from each other, every single day.
DOCUMENT Employee Handbook — All Employees
VERSION Version 6.0  |  Effective 1 April 2026
APPLIES TO All full-time, part-time, contract and intern staff in India
OWNER Human Resources Department, PROITBRIDGE
REVIEW CYCLE Annually, or on any change in applica


## 6.Generate Embeddings

Embeddings convert each chunk of text into a numerical vector that respresents its semantic meaning.

In [14]:
EMBEDDING_MODEL = "text-embedding-3-small"

def get_embeddings(texts,batch_size=100):
    all_embeddings=[]

    for start in range(0,len(texts),batch_size):
        batch=texts[start:start+batch_size]

        response = client.embeddings.create(model=EMBEDDING_MODEL,input=batch)

        ordered=sorted(response.data,key=lambda item: item.index)
        all_embeddings.extend([item.embedding for item in ordered])

    
    return all_embeddings

chunk_texts =[chunk["text"] for chunk in chunks]
chunk_embeddings=get_embeddings(chunk_texts)

print("Number of Embeddings:",len(chunk_embeddings))
print("Embedding Dimensions:",len(chunk_embeddings[0]))
print("First 10 Values",chunk_embeddings[0][:10])

Number of Embeddings: 233
Embedding Dimensions: 1536
First 10 Values [0.0103759765625, 0.016021728515625, 0.051544189453125, 0.0234832763671875, -0.0078582763671875, -0.002841949462890625, 0.0303955078125, 0.02142333984375, -0.022491455078125, 0.0011453628540039062]


## 7.Store in Vector Database

In [18]:
import chromadb

chroma_client = chromadb.PersistentClient(path=".\chroma_db")
collection=chroma_client.get_or_create_collection(name="employee_handbook")

#Clearing if ther's any collection.So,that rerunning this notebook doesn't create duplicates

existing = collection.get()

if existing["ids"]:

    collection.delete(ids=exisitng["ids"])


collection.add(ids=[chunk["id"] for chunk in chunks],
                documents=[chunk["text"] for chunk in chunks],
                embeddings=chunk_embeddings,
                metadatas=[{"source":pdf_path} for _ in chunks])

print("Chunks stored in vector Database:",collection.count())

Chunks stored in vector Database: 233


## 8.User Query -> Query Embedding

In [19]:
question = "How many casusal leaves are allowed?"

question_embedding = get_embeddings([question])[0]

print("Question:",question)
print("Query Embedding Dimensions:",len(question_embedding))

Question: How many casusal leaves are allowed?
Query Embedding Dimensions: 1536


## 9.Retriever -> Search the vector database

In [24]:
results = collection.query(query_embeddings=[question_embedding],n_results=3)

retrieved_chunks = results["documents"][0]
retrieved_ids = results["ids"][0]
distances=results["distances"][0]
print("Retrieved Chunks:\n")

for i,(chunk_id,chunk_text,distance) in enumerate(zip(retrieved_ids,retrieved_chunks,distances),start=1):
    print(f"---Result {i} | {chunk_id} | distance={distance:.4f} ---- ")
    print(chunk_text[:100])
    print()

Retrieved Chunks:

---Result 1 | chunk-70 | distance=0.9776 ---- 
year. Anything above 45 days lapses on 31 December unless
leave was refused in writing for business 

---Result 2 | chunk-68 | distance=1.0262 ---- 
t On qualifying event Not applicable No
Marriage Leave 5 days, once in
service
On qualifying event N

---Result 3 | chunk-84 | distance=1.0569 ---- 
olidays are counted
as leave. For CL and SL they are not.
Can leave be cancelled after
approval?
Yes



In [25]:
results

{'ids': [['chunk-70', 'chunk-68', 'chunk-84']],
 'embeddings': None,
 'documents': [['year. Anything above 45 days lapses on 31 December unless\nleave was refused in writing for business reasons.\nEncashed at basic salary on separation, for the balance standing to your credit.\n7.3 Casual Leave (CL)\nFor short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.\nMaximum 3 consecutive days at a time. It cannot be combined with EL.\n• \n• \n• \n• \n• \n• \n• \n• \nPROITBRIDGE Employee Handbook\nVersion 6.0 \xa0|\xa0 Effective 1 April 2026 \xa0|\xa0 Internal & Confidential',
   't On qualifying event Not applicable No\nMarriage Leave 5 days, once in\nservice\nOn qualifying event Not applicable No\nCompensatory Off As earned On approved extra\nwork\n90 days validity No\nLeave Without Pay (LWP) Case by case On approval Not applicable No\nLeave year\nThe leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-\nrated

## 10. Prompt + Retrieved Context

Now we combine the user's question with the relevant chunks returned by the retriever.

This retrieved context becomes part of the input sent to LL<.

In [26]:
retrieved_chunks

['year. Anything above 45 days lapses on 31 December unless\nleave was refused in writing for business reasons.\nEncashed at basic salary on separation, for the balance standing to your credit.\n7.3 Casual Leave (CL)\nFor short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.\nMaximum 3 consecutive days at a time. It cannot be combined with EL.\n• \n• \n• \n• \n• \n• \n• \n• \nPROITBRIDGE Employee Handbook\nVersion 6.0 \xa0|\xa0 Effective 1 April 2026 \xa0|\xa0 Internal & Confidential',
 't On qualifying event Not applicable No\nMarriage Leave 5 days, once in\nservice\nOn qualifying event Not applicable No\nCompensatory Off As earned On approved extra\nwork\n90 days validity No\nLeave Without Pay (LWP) Case by case On approval Not applicable No\nLeave year\nThe leave year runs from 1 January to 31 December. Employees joining mid-year receive CL and SL pro-\nrated to the number of completed months remaining in the year.\n7.2 Earned Leave (EL / Priv

In [27]:
context = "\n\n--- Retrieved Chunk ---\n\n".join(retrieved_chunks)

prompt =f"""Answer the user's question using ONLY the retreieved context below.
          If the answer is not present in the context, say that the information is not available in the provided document.
          Retrieved Context:{context}
          User Question:{question}""".strip()

print(prompt)


Answer the user's question using ONLY the retreieved context below.
          If the answer is not present in the context, say that the information is not available in the provided document.
          Retrieved Context:year. Anything above 45 days lapses on 31 December unless
leave was refused in writing for business reasons.
Encashed at basic salary on separation, for the balance standing to your credit.
7.3 Casual Leave (CL)
For short, unplanned personal needs — a bank appointment, a family obligation, a delayed commute.
Maximum 3 consecutive days at a time. It cannot be combined with EL.
• 
• 
• 
• 
• 
• 
• 
• 
PROITBRIDGE Employee Handbook
Version 6.0  |  Effective 1 April 2026  |  Internal & Confidential

--- Retrieved Chunk ---

t On qualifying event Not applicable No
Marriage Leave 5 days, once in
service
On qualifying event Not applicable No
Compensatory Off As earned On approved extra
work
90 days validity No
Leave Without Pay (LWP) Case by case On approval Not applicable No
L

## 11.Generate the Answer - OpenAI Responses API

In [32]:
GENERATION_MODEL ="gpt-4.1-mini"

response=client.responses.create(model=GENERATION_MODEL,input = prompt)
answer=response.output_text

print(question)
print("Answer:\n")

print(answer)

How many casusal leaves are allowed?
Answer:

The number of casual leaves (CL) allowed is for short, unplanned personal needs, with a maximum of 3 consecutive days at a time.


# 2 Points to Note

## 1. The role Of the vector database?

### 

- The Vector database doesn't generate the answer.

- It's Job is to store  the embeddings and retrieve the most relevant chunks for teh user query.

### 2.The role of the LLM

- the LLM receives the retrieved context and generates the final answer